# In-class activity

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.utils.data as data
import math
import copy

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super(MultiHeadAttention, self).__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"

        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads

        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

    def scaled_dot_product_attention(self, Q, K, V, mask=None):
        attn_scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        if mask is not None:
            attn_scores = attn_scores.masked_fill(mask == 0, -1e9)
        attn_probs = torch.softmax(attn_scores, dim=-1)
        output = torch.matmul(attn_probs, V)
        return output

    def split_heads(self, x):
        batch_size, seq_length, d_model = x.size()
        return x.view(batch_size, seq_length, self.num_heads, self.d_k).transpose(1, 2)

    def combine_heads(self, x):
        batch_size, _, seq_length, d_k = x.size()
        return x.transpose(1, 2).contiguous().view(batch_size, seq_length, self.d_model)

    def forward(self, Q, K, V, mask=None):
        Q = self.split_heads(self.W_q(Q))
        K = self.split_heads(self.W_k(K))
        V = self.split_heads(self.W_v(V))

        attn_output = self.scaled_dot_product_attention(Q, K, V, mask)
        output = self.W_o(self.combine_heads(attn_output))
        return output

In [ ]:
class PositionWiseFeedForward(nn.Module):
    def __init__(self, d_model, d_ff):
        super(PositionWiseFeedForward, self).__init__()
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)
        self.relu = nn.ReLU()

    def forward(self, x):
        # TODO: Pass the input through the fully-connected layers (~1 line of code)
        raise NotImplementedError
        # END OF TODO

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_seq_length):
        super(PositionalEncoding, self).__init__()

        pe = torch.zeros(max_seq_length, d_model)
        position = torch.arange(0, max_seq_length, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * -(math.log(10000.0) / d_model))

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        # TODO: Add positional encoding to the input. (~1 line of code)
        raise NotImplementedError
        # END OF TODO

In [ ]:
class EncoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout):
        super(EncoderLayer, self).__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads)
        self.feed_forward = PositionWiseFeedForward(d_model, d_ff)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask):
        # TODO: Do the following steps: (~7 lines of code)
        #   1. Multi-headed attention
        #   2. Dropout on the attention output
        #   3. Add & Norm
        #   4. Feed forward
        #   5. Dropout on the feed forward output
        #   6. Add & Norm
        raise NotImplementedError
        # END OF TODO

In [ ]:
class DecoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout):
        super(DecoderLayer, self).__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads)
        self.cross_attn = MultiHeadAttention(d_model, num_heads)
        self.feed_forward = PositionWiseFeedForward(d_model, d_ff)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, enc_output, src_mask, tgt_mask):
        # TODO: Do the following steps: (~10 lines of code)
        #   1. Masked multi-headed attention
        #   2. Dropout on the attention output
        #   3. Add & Norm
        #   4. Multi-headed cross attention
        #   5. Dropout on the cross attention output
        #   6. Add & Norm
        #   7. Feed forward
        #   8. Dropout on the feed forward output
        #   9. Add & Norm
        raise NotImplementedError
        # END OF TODO

In [ ]:
class Transformer(nn.Module):
    def __init__(self, src_vocab_size, tgt_vocab_size, d_model, num_heads, num_layers, d_ff, max_seq_length, dropout):
        super(Transformer, self).__init__()
        self.encoder_embedding = nn.Embedding(src_vocab_size, d_model)
        self.decoder_embedding = nn.Embedding(tgt_vocab_size, d_model)
        self.positional_encoding = PositionalEncoding(d_model, max_seq_length)

        # TODO: Define encoder and decoder layers. (~2 lines of code)
        # (Hint: you can use nn.ModuleList to stack some layers on top of each other)
        self.encoder_layers = None
        self.decoder_layers = None
        # END OF TODO

        self.fc = nn.Linear(d_model, tgt_vocab_size)
        self.dropout = nn.Dropout(dropout)

    def generate_mask(self, src, tgt):
        src_mask = (src != 0).unsqueeze(1).unsqueeze(2)
        tgt_mask = (tgt != 0).unsqueeze(1).unsqueeze(3)
        seq_length = tgt.size(1)
        nopeak_mask = (1 - torch.triu(torch.ones(1, seq_length, seq_length), diagonal=1)).bool().to(tgt_mask.device)
        tgt_mask = tgt_mask & nopeak_mask
        return src_mask, tgt_mask

    def forward(self, src, tgt):
        src_mask, tgt_mask = self.generate_mask(src, tgt)
        src_embedded = self.dropout(self.positional_encoding(self.encoder_embedding(src)))
        tgt_embedded = self.dropout(self.positional_encoding(self.decoder_embedding(tgt)))

        enc_output = src_embedded
        for enc_layer in self.encoder_layers:
            # TODO: Call encoder layers (~1 line of code)
            enc_output = None
            # END OF TODO

        dec_output = tgt_embedded
        for dec_layer in self.decoder_layers:
            # TODO: Call decoder layers (~1 line of code)
            dec_output = None
            # END OF TODO

        output = self.fc(dec_output)
        return output

In [ ]:
src_vocab_size = 5000
tgt_vocab_size = 5000
d_model = 512
num_heads = 8
num_layers = 6
d_ff = 2048
max_seq_length = 100
dropout = 0.1

transformer = Transformer(src_vocab_size, tgt_vocab_size, d_model, num_heads, num_layers, d_ff, max_seq_length, dropout).cuda()

# Generate random sample data
src_data = torch.randint(1, src_vocab_size, (64, max_seq_length))  # (batch_size, seq_length)
tgt_data = torch.randint(1, tgt_vocab_size, (64, max_seq_length))  # (batch_size, seq_length)

In [ ]:
criterion = nn.CrossEntropyLoss(ignore_index=0)
optimizer = optim.Adam(transformer.parameters(), lr=0.0001, betas=(0.9, 0.98), eps=1e-9)

transformer.train()

for epoch in range(100):
    optimizer.zero_grad()
    src_data = src_data.cuda()
    tgt_data = tgt_data.cuda()
    output = transformer(src_data, tgt_data[:, :-1])
    loss = criterion(output.contiguous().view(-1, tgt_vocab_size), tgt_data[:, 1:].contiguous().view(-1))
    loss.backward()
    optimizer.step()
    print(f"Epoch: {epoch+1}, Loss: {loss.item()}")

# Homework (Please run the previous section first)

In [ ]:
!pip install datasets

In [ ]:
from datasets import load_dataset

# TODO: Load first 10240 samples from training part of text2log dataset from HuggingFace (~1 line of code)
text2log_dataset = None
# END OF TODO
print(text2log_dataset)

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

def preprocess_data(samples):
    # TODO: Use BERT tokenizer to tokenize "sentence" and "fol_translation" (~2 lines of code)
    # (Hint: you should pad samples to max length, truncate longer ones and return pytorch tensor)
    preprocessed_data = None
    fol_translation_tokenized = None
    # END OF TODO
    preprocessed_data['target_input_ids'] = fol_translation_tokenized['input_ids']
    preprocessed_data['target_attention_mask'] = fol_translation_tokenized['attention_mask']

    return preprocessed_data

tokenized_dataset = text2log_dataset.map(preprocess_data, batched=True, batch_size=5120)

In [ ]:
# TODO: Set right vocabualry sizes (~2 lines of code)
# (Hint: you can use tokenizer)
src_vocab_size = None
tgt_vocab_size = None
# END OF TODO
d_model = 512
num_heads = 8
num_layers = 6
d_ff = 2048
max_seq_length = 512
dropout = 0.1

transformer = Transformer(src_vocab_size, tgt_vocab_size, d_model, num_heads, num_layers, d_ff, max_seq_length, dropout).cuda()

In [ ]:
def get_batch(dataset, batch_size, start_index):
    # TODO: Return the batch starting from start index (~1 line)
    # WARNING: MAKE SURE YOUR CODE WORKS FINE WITH THE LAST BATCH!!!
    raise NotImplementedError
    #end of TODO

In [ ]:
from tqdm import tqdm

criterion = nn.CrossEntropyLoss(ignore_index=0)
optimizer = optim.Adam(transformer.parameters(), lr=0.0001, betas=(0.9, 0.98), eps=1e-9)

transformer.train()

batch_size = 16

tokenized_dataset.set_format(type='torch', columns=['input_ids', 'target_input_ids'])
for epoch in range(3):
    with tqdm(range(0, tokenized_dataset.num_rows, batch_size), desc=f"Epoch {epoch+1}", postfix={"Loss": 0}) as pbar:
        for start_index in pbar:
            optimizer.zero_grad()
            batch = get_batch(tokenized_dataset, batch_size, start_index)[:]
            src_data = batch['input_ids'].cuda()
            tgt_data = batch['target_input_ids'].cuda()
            output = transformer(src_data, tgt_data[:, :-1])
            loss = criterion(output.contiguous().view(-1, tgt_vocab_size), tgt_data[:, 1:].contiguous().view(-1))
            loss.backward()
            optimizer.step()
            pbar.set_postfix({"Loss": loss.item()})